# 02 — Chunking 实验

对比三种切分策略在不同 chunk_size / overlap 下的 chunk 差异。

In [ ]:
import sys
sys.path.insert(0, '..')
import logging
logging.basicConfig(level=logging.WARNING, format='%(levelname)s:%(message)s')

from src.loader import load_directory
from src.chunking import get_chunker

In [ ]:
# 加载文档
docs = load_directory('../data/raw')
full_text = '\n\n'.join(d.text for d in docs)
print(f'Total text length: {len(full_text)} chars')

## 实验 E1~E5

| 实验 | strategy | chunk_size | overlap |
|------|----------|------------|--------|
| E1 | sentence | 512 | 128 |
| E2 | jieba | 512 | 128 |
| E3 | semantic | auto | 0 |
| E4 | sentence | 256 | 64 |
| E5 | sentence | 1024 | 256 |

In [ ]:
experiments = [
    ("E1", "sentence", 512, 128),
    ("E2", "jieba", 512, 128),
    ("E3", "sentence", 256, 64),
    ("E4", "sentence", 1024, 256),
]

for name, strategy, size, overlap in experiments:
    chunker = get_chunker(strategy, chunk_size=size, chunk_overlap=overlap)
    chunks = chunker.chunk(full_text, metadata={"source": "all"})
    print(f'{name} ({strategy}, size={size}, overlap={overlap}): {len(chunks)} chunks')
    for c in chunks[:3]:
        print(f'  [{c.index}] {len(c.text)} chars: {c.text[:100]}...')

## 观察

- 不同策略产生的 chunk 数量差异
- 有无语义被切碎的情况
- 每个 chunk 的信息完整度

In [ ]:
# 对比不同策略的第一个 chunk 看切分位置
from src.chunking import SentenceChunker, JiebaChunker

for name, cls in [('sentence', SentenceChunker), ('jieba', JiebaChunker)]:
    c = cls(chunk_size=512, chunk_overlap=128)
    chunks = c.chunk(full_text, metadata={"source": "all"})
    print(f'=== {name}: first chunk (end) ===')
    print(chunks[0].text[-150:])
    print(f'=== {name}: second chunk (start) ===')
    if len(chunks) > 1:
        print(chunks[1].text[:150])
    print()